In [ ]:
# ==============================
# STEP 1: Upload final datasets
# ==============================

from google.colab import files

uploaded = files.upload()

Saving cleaned_cv_dataset.json to cleaned_cv_dataset.json


In [ ]:
# ==============================
# STEP 2: Import libraries
# ==============================

import pandas as pd
import numpy as np
import json
import ast
import re
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import joblib

In [ ]:
# ==============================
# STEP 3: Load job post and CV datasets
# ==============================

job_file = "cleaned_job_posts_dataset.csv"
cv_file = "cleaned_cv_dataset.json"

job_df = pd.read_csv(job_file)

with open(cv_file, "r", encoding="utf-8") as file:
    cv_data = json.load(file)

cv_df = pd.DataFrame(cv_data)

print("Job post dataset shape:", job_df.shape)
print("CV dataset shape:", cv_df.shape)

print("\nJob post columns:")
print(job_df.columns.tolist())

print("\nCV columns:")
print(cv_df.columns.tolist())

Job post dataset shape: (2400, 13)
CV dataset shape: (2066, 32)

Job post columns:
['id', 'title', 'company', 'tags', 'description', 'cleaned_description', 'extracted_skills', 'description_word_count', 'title_word_count', 'post_date', 'week', 'scraped_at', 'target_role']

CV columns:
['candidate_id', 'current_role', 'target_role', 'current_employment', 'degree', 'university', 'education_year', 'cleaned_all_skills', 'skills_text', 'num_cleaned_skills', 'programming_languages', 'frameworks', 'technologies', 'databases', 'technical_skills', 'soft_skills', 'experience_months', 'experience_level', 'num_projects', 'has_projects', 'project_names', 'project_names_text', 'project_technologies', 'project_technologies_text', 'project_complexities', 'cleaned_certificates', 'certificates_text', 'num_certificates', 'has_certificates', 'evaluation_score', 'skill_score', 'combined_cv_text']


In [ ]:
# ==============================
# STEP 4: Create helper functions
# ==============================

def clean_text(text):
    if text is None:
        return ""

    if isinstance(text, float) and pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


def parse_list(value):
    if isinstance(value, list):
        return value

    if value is None:
        return []

    if isinstance(value, float) and pd.isna(value):
        return []

    value = str(value).strip()

    if value == "":
        return []

    try:
        parsed_value = ast.literal_eval(value)

        if isinstance(parsed_value, list):
            return parsed_value
    except:
        pass

    return [item.strip() for item in value.split(",") if item.strip() != ""]


def normalize_skill(skill):
    skill = clean_text(skill).lower()

    skill = skill.replace(".", "")
    skill = skill.replace("-", " ")
    skill = skill.replace("_", " ")
    skill = re.sub(r"\s+", " ", skill).strip()

    skill_mapping = {
        "pytorch": "pytorch",
        "py torch": "pytorch",

        "tensorflow": "tensorflow",
        "tensor flow": "tensorflow",

        "scikit learn": "scikit learn",
        "sklearn": "scikit learn",

        "fastapi": "fastapi",
        "fast api": "fastapi",

        "rest api": "rest api",
        "restful api": "rest api",

        "nodejs": "nodejs",
        "node js": "nodejs",

        "reactjs": "react",
        "react js": "react",

        "nextjs": "nextjs",
        "next js": "nextjs",

        "ci cd": "ci cd",
        "cicd": "ci cd",

        "sqlite": "sqlite",
        "sql lite": "sqlite",

        "vs code": "vs code",
        "vscode": "vs code",

        "powerbi": "power bi",
        "power bi": "power bi",

        "javascript": "javascript",
        "java script": "javascript",

        "postgres": "postgresql",
        "postgresql": "postgresql",

        "mongodb": "mongodb",
        "mongo db": "mongodb",

        "mysql": "mysql",
        "my sql": "mysql",

        "opencv": "opencv",
        "open cv": "opencv",

        "nlp": "nlp",
        "natural language processing": "nlp",

        "machine learning": "machine learning",
        "deep learning": "deep learning"
    }

    if skill in skill_mapping:
        return skill_mapping[skill]

    return skill


def display_skill(skill):
    display_mapping = {
        "pytorch": "PyTorch",
        "tensorflow": "TensorFlow",
        "scikit learn": "Scikit-learn",
        "fastapi": "FastAPI",
        "rest api": "REST API",
        "nodejs": "Node.js",
        "react": "React",
        "nextjs": "Next.js",
        "ci cd": "CI/CD",
        "sqlite": "SQLite",
        "vs code": "VS Code",
        "power bi": "Power BI",
        "javascript": "JavaScript",
        "postgresql": "PostgreSQL",
        "mongodb": "MongoDB",
        "mysql": "MySQL",
        "opencv": "OpenCV",
        "nlp": "NLP",
        "machine learning": "Machine Learning",
        "deep learning": "Deep Learning"
    }

    return display_mapping.get(skill, skill.title())


def get_level(score):
    if score >= 80:
        return "Advanced"
    elif score >= 50:
        return "Intermediate"
    else:
        return "Beginner"

In [ ]:
# ==============================
# STEP 5: Create job role profiles
# ==============================

job_role_profiles = {}

for role, role_df in job_df.groupby("target_role"):
    skill_counter = Counter()

    for skills_value in role_df["extracted_skills"]:
        skills = parse_list(skills_value)

        for skill in skills:
            normalized_skill = normalize_skill(skill)

            if normalized_skill != "":
                skill_counter[normalized_skill] += 1

    most_common_skills = skill_counter.most_common()

    required_skills = [
        display_skill(skill)
        for skill, count in most_common_skills[:10]
    ]

    preferred_skills = [
        display_skill(skill)
        for skill, count in most_common_skills[10:25]
    ]

    tools_keywords = [
        "power bi", "tableau", "excel", "git", "github", "docker",
        "kubernetes", "aws", "azure", "gcp", "jira", "postman",
        "figma", "vs code", "jupyter"
    ]

    common_tools = []

    for skill, count in most_common_skills:
        if skill in tools_keywords:
            common_tools.append(display_skill(skill))

    job_role_profiles[role] = {
        "role": role,
        "total_job_posts": int(len(role_df)),
        "required_skills": required_skills,
        "preferred_skills": preferred_skills,
        "common_tools": common_tools[:10]
    }

with open("job_role_profiles.json", "w", encoding="utf-8") as file:
    json.dump(job_role_profiles, file, indent=4, ensure_ascii=False)

print("job_role_profiles.json created successfully")
print("Total roles:", len(job_role_profiles))

list(job_role_profiles.items())[:2]

job_role_profiles.json created successfully
Total roles: 24


[('AI Engineer',
  {'role': 'AI Engineer',
   'total_job_posts': 100,
   'required_skills': ['Python',
    'Aws',
    'Docker',
    'NLP',
    'Git',
    'Kubernetes',
    'FastAPI',
    'React',
    'Typescript',
    'Java'],
   'preferred_skills': ['JavaScript',
    'Azure',
    'Github',
    'Gcp',
    'Node.js',
    'Agile',
    'Machine Learning',
    'Sql',
    'Product Management',
    'Linux',
    'Css',
    'Html',
    'Express',
    'MySQL',
    'Excel'],
   'common_tools': ['Aws',
    'Docker',
    'Git',
    'Kubernetes',
    'Azure',
    'Github',
    'Gcp',
    'Excel',
    'Jira']}),
 ('AI ML',
  {'role': 'AI ML',
   'total_job_posts': 100,
   'required_skills': ['Python',
    'NLP',
    'Deep Learning',
    'Computer Vision',
    'Machine Learning',
    'Scikit-learn',
    'Pandas',
    'PyTorch',
    'Numpy',
    'TensorFlow'],
   'preferred_skills': ['Kubernetes',
    'Docker',
    'Git',
    'Data Engineering',
    'Github',
    'Jira',
    'Java',
    'React',
    '

In [ ]:
# ==============================
# STEP 5.1: Download job role profiles
# ==============================

files.download("job_role_profiles.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================
# STEP 6: Compare each CV with target role profile
# ==============================

def compare_cv_with_role(cv_row, job_role_profiles):
    target_role = clean_text(cv_row.get("target_role", ""))

    profile = job_role_profiles.get(target_role, None)

    if profile is None:
        return {
            "matched_required_skills": [],
            "matched_preferred_skills": [],
            "missing_required_skills": [],
            "missing_preferred_skills": [],
            "required_match_ratio": 0,
            "preferred_match_ratio": 0,
            "project_match_ratio": 0
        }

    cv_skills = parse_list(cv_row.get("cleaned_all_skills", []))
    project_skills = parse_list(cv_row.get("project_technologies", []))

    cv_skill_set = set([normalize_skill(skill) for skill in cv_skills])
    project_skill_set = set([normalize_skill(skill) for skill in project_skills])

    required_skills = profile["required_skills"]
    preferred_skills = profile["preferred_skills"]

    required_skill_set = set([normalize_skill(skill) for skill in required_skills])
    preferred_skill_set = set([normalize_skill(skill) for skill in preferred_skills])

    matched_required = cv_skill_set.intersection(required_skill_set)
    matched_preferred = cv_skill_set.intersection(preferred_skill_set)

    missing_required = required_skill_set.difference(cv_skill_set)
    missing_preferred = preferred_skill_set.difference(cv_skill_set)

    role_all_skills = required_skill_set.union(preferred_skill_set)
    matched_project_skills = project_skill_set.intersection(role_all_skills)

    required_match_ratio = len(matched_required) / len(required_skill_set) if len(required_skill_set) > 0 else 0
    preferred_match_ratio = len(matched_preferred) / len(preferred_skill_set) if len(preferred_skill_set) > 0 else 0
    project_match_ratio = len(matched_project_skills) / len(role_all_skills) if len(role_all_skills) > 0 else 0

    return {
        "matched_required_skills": [display_skill(skill) for skill in sorted(matched_required)],
        "matched_preferred_skills": [display_skill(skill) for skill in sorted(matched_preferred)],
        "missing_required_skills": [display_skill(skill) for skill in sorted(missing_required)],
        "missing_preferred_skills": [display_skill(skill) for skill in sorted(missing_preferred)],
        "required_match_ratio": required_match_ratio,
        "preferred_match_ratio": preferred_match_ratio,
        "project_match_ratio": project_match_ratio
    }

In [ ]:
# ==============================
# STEP 7: Generate match score and level
# ==============================

scored_rows = []

for index, row in cv_df.iterrows():
    comparison = compare_cv_with_role(row, job_role_profiles)

    experience_months = float(row.get("experience_months", 0))
    num_projects = float(row.get("num_projects", 0))
    num_certificates = float(row.get("num_certificates", 0))
    evaluation_score = float(row.get("evaluation_score", 0))

    skill_score_component = (
        comparison["required_match_ratio"] * 40 +
        comparison["preferred_match_ratio"] * 10
    )

    project_score_component = (
        comparison["project_match_ratio"] * 15 +
        min(num_projects / 3, 1) * 5
    )

    experience_score_component = min(experience_months / 24, 1) * 15

    certificate_score_component = min(num_certificates / 5, 1) * 10

    ats_score_component = min(evaluation_score, 100) / 100 * 5

    final_score = (
        skill_score_component +
        project_score_component +
        experience_score_component +
        certificate_score_component +
        ats_score_component
    )

    final_score = round(final_score, 2)

    level = get_level(final_score)

    scored_rows.append({
        "candidate_id": row.get("candidate_id", ""),
        "target_role": row.get("target_role", ""),
        "experience_level": row.get("experience_level", ""),

        "cv_skills_count": len(parse_list(row.get("cleaned_all_skills", []))),
        "required_skills_count": len(job_role_profiles.get(row.get("target_role", ""), {}).get("required_skills", [])),
        "preferred_skills_count": len(job_role_profiles.get(row.get("target_role", ""), {}).get("preferred_skills", [])),

        "matched_required_skills": comparison["matched_required_skills"],
        "matched_preferred_skills": comparison["matched_preferred_skills"],
        "missing_required_skills": comparison["missing_required_skills"],
        "missing_preferred_skills": comparison["missing_preferred_skills"],

        "matched_required_count": len(comparison["matched_required_skills"]),
        "matched_preferred_count": len(comparison["matched_preferred_skills"]),
        "missing_required_count": len(comparison["missing_required_skills"]),
        "missing_preferred_count": len(comparison["missing_preferred_skills"]),

        "required_match_ratio": comparison["required_match_ratio"],
        "preferred_match_ratio": comparison["preferred_match_ratio"],
        "project_match_ratio": comparison["project_match_ratio"],

        "experience_months": experience_months,
        "num_projects": num_projects,
        "has_projects": int(row.get("has_projects", 0)),
        "num_certificates": num_certificates,
        "has_certificates": int(row.get("has_certificates", 0)),
        "ats_quality_score": min(evaluation_score, 100),

        "skill_score_component": round(skill_score_component, 2),
        "project_score_component": round(project_score_component, 2),
        "experience_score_component": round(experience_score_component, 2),
        "certificate_score_component": round(certificate_score_component, 2),
        "ats_score_component": round(ats_score_component, 2),

        "final_score": final_score,
        "level": level
    })

scored_df = pd.DataFrame(scored_rows)

print("Scoring completed successfully")
print("Scored dataset shape:", scored_df.shape)

scored_df.head()

Scoring completed successfully
Scored dataset shape: (2066, 30)


,candidate_id,target_role,experience_level,cv_skills_count,required_skills_count,preferred_skills_count,matched_required_skills,matched_preferred_skills,missing_required_skills,missing_preferred_skills,...,num_certificates,has_certificates,ats_quality_score,skill_score_component,project_score_component,experience_score_component,certificate_score_component,ats_score_component,final_score,level
0,CAND_920D187A,AI Engineer,Fresher,22,10,15,"[Java, NLP, Python]","[Css, Html, JavaScript, Sql]","[Aws, Docker, FastAPI, Git, Kubernetes, React,...","[Agile, Azure, Excel, Express, Gcp, Github, Li...",...,12.0,1,36.0,14.67,5.60,0.0,10.0,1.80,32.07,Beginner
1,CAND_B208BBBB,Data Analyst,Junior,32,10,7,"[Analytical Thinking, Jupyter Notebook, Power BI]",[Python],"[Attention To Detail, Communication, Data Anal...","[Dashboard Development, Google Sheets, Pandas,...",...,0.0,0,52.0,13.43,4.22,7.5,0.0,2.60,27.74,Beginner
2,CAND_15E5E1DE,Software Developer,Fresher,24,10,15,"[Java, JavaScript, Python]","[Css, Html, MySQL, Node.js]","[Angular, Aws, Docker, Kubernetes, React, Sql,...","[Agile, Azure, Data Engineering, Figma, Flask,...",...,0.0,0,49.0,14.67,7.40,0.0,0.0,2.45,24.52,Beginner
3,CAND_B41CA43C,Software Developer,Fresher,16,10,15,"[Java, JavaScript, Python, React]","[Css, Git, Html, MySQL]","[Angular, Aws, Docker, Kubernetes, Sql, Typesc...","[Agile, Azure, Data Engineering, Figma, Flask,...",...,0.0,0,44.5,18.67,4.53,0.0,0.0,2.23,25.43,Beginner
4,CAND_D41C2E0F,Data Scientist,Fresher,47,10,9,"[Pandas, Python, Scikit-learn]","[Data Visualization, Matplotlib, Problem Solvi...","[Communication, Curiosity, Data Preprocessing,...","[Analytical Thinking, Data Science, Jupyter No...",...,3.0,1,41.0,16.44,7.37,0.0,6.0,2.05,31.86,Beginner


In [ ]:
# ==============================
# STEP 8: Save labeled training dataset
# ==============================

training_file = "scored_training_dataset.csv"

scored_df.to_csv(training_file, index=False)

print("Labeled training dataset saved successfully")
print("File name:", training_file)

files.download(training_file)

Labeled training dataset saved successfully
File name: scored_training_dataset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================
# STEP 9: Check score and level distribution
# ==============================

print("Level distribution:")
print(scored_df["level"].value_counts())

print("\nScore summary:")
print(scored_df["final_score"].describe())

print("\nAverage score by target role:")
print(scored_df.groupby("target_role")["final_score"].mean().sort_values(ascending=False))

Level distribution:
level
Beginner        1891
Intermediate     175
Name: count, dtype: int64

Score summary:
count    2066.000000
mean       30.836254
std        12.673939
min         0.000000
25%        21.232500
50%        29.445000
75%        39.675000
max        69.030000
Name: final_score, dtype: float64

Average score by target role:
target_role
AI ML                                         47.260507
Full Stack Developer                          46.139140
Software Engineer                             44.388689
Data Scientist                                38.267876
Data Analyst                                  38.085977
Web Developer / Software Quality Assurance    37.310000
Quality Engineer                              37.300000
AI Engineer                                   32.070000
Backend Developer                             31.024107
Mobile Application Developer                  30.370000
Data Engineer                                 28.724636
Machine Learning Engineer    

In [ ]:
# ==============================
# STEP 10: Train score prediction model
# ==============================

feature_columns = [
    "target_role",
    "experience_level",

    "cv_skills_count",
    "required_skills_count",
    "preferred_skills_count",
    "matched_required_count",
    "matched_preferred_count",
    "missing_required_count",
    "missing_preferred_count",

    "required_match_ratio",
    "preferred_match_ratio",
    "project_match_ratio",

    "experience_months",
    "num_projects",
    "has_projects",
    "num_certificates",
    "has_certificates",
    "ats_quality_score"
]

target_column = "final_score"

X = scored_df[feature_columns]
y = scored_df[target_column]

categorical_features = [
    "target_role",
    "experience_level"
]

numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

score_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42
        ))
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

score_model.fit(X_train, y_train)

y_pred = score_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model trained successfully")
print("Mean Absolute Error:", round(mae, 2))
print("R2 Score:", round(r2, 4))

Model trained successfully
Mean Absolute Error: 1.22
R2 Score: 0.9851


In [ ]:
# ==============================
# STEP 11: Save trained model
# ==============================

model_file = "cv_job_score_model.pkl"

joblib.dump(score_model, model_file)

print("Model saved successfully")
print("File name:", model_file)

files.download(model_file)

Model saved successfully
File name: cv_job_score_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================
# STEP 12: Create prediction function
# ==============================

def build_feature_row_for_prediction(cv_row, selected_role, job_role_profiles):
    cv_row = cv_row.copy()
    cv_row["target_role"] = selected_role

    comparison = compare_cv_with_role(cv_row, job_role_profiles)

    experience_months = float(cv_row.get("experience_months", 0))
    num_projects = float(cv_row.get("num_projects", 0))
    num_certificates = float(cv_row.get("num_certificates", 0))
    evaluation_score = float(cv_row.get("evaluation_score", 0))

    feature_row = {
        "target_role": selected_role,
        "experience_level": cv_row.get("experience_level", ""),

        "cv_skills_count": len(parse_list(cv_row.get("cleaned_all_skills", []))),
        "required_skills_count": len(job_role_profiles.get(selected_role, {}).get("required_skills", [])),
        "preferred_skills_count": len(job_role_profiles.get(selected_role, {}).get("preferred_skills", [])),
        "matched_required_count": len(comparison["matched_required_skills"]),
        "matched_preferred_count": len(comparison["matched_preferred_skills"]),
        "missing_required_count": len(comparison["missing_required_skills"]),
        "missing_preferred_count": len(comparison["missing_preferred_skills"]),

        "required_match_ratio": comparison["required_match_ratio"],
        "preferred_match_ratio": comparison["preferred_match_ratio"],
        "project_match_ratio": comparison["project_match_ratio"],

        "experience_months": experience_months,
        "num_projects": num_projects,
        "has_projects": int(cv_row.get("has_projects", 0)),
        "num_certificates": num_certificates,
        "has_certificates": int(cv_row.get("has_certificates", 0)),
        "ats_quality_score": min(evaluation_score, 100)
    }

    return feature_row, comparison


def predict_cv_readiness(cv_row, selected_role):
    feature_row, comparison = build_feature_row_for_prediction(
        cv_row,
        selected_role,
        job_role_profiles
    )

    feature_df = pd.DataFrame([feature_row])

    predicted_score = score_model.predict(feature_df)[0]
    predicted_score = round(float(predicted_score), 2)

    predicted_level = get_level(predicted_score)

    recommendations = []

    for skill in comparison["missing_required_skills"][:5]:
        recommendations.append("Learn or improve " + skill)

    return {
        "candidate_id": cv_row.get("candidate_id", ""),
        "selected_role": selected_role,
        "predicted_score": predicted_score,
        "predicted_level": predicted_level,
        "matched_required_skills": comparison["matched_required_skills"],
        "missing_required_skills": comparison["missing_required_skills"],
        "missing_preferred_skills": comparison["missing_preferred_skills"],
        "recommendations": recommendations
    }

In [ ]:
# ==============================
# STEP 13: Test prediction with one CV
# ==============================

sample_cv = cv_df.iloc[0].to_dict()
selected_role = sample_cv["target_role"]

result = predict_cv_readiness(sample_cv, selected_role)

result

{'candidate_id': 'CAND_920D187A',
 'selected_role': 'AI Engineer',
 'predicted_score': 30.45,
 'predicted_level': 'Beginner',
 'matched_required_skills': ['Java', 'NLP', 'Python'],
 'missing_required_skills': ['Aws',
  'Docker',
  'FastAPI',
  'Git',
  'Kubernetes',
  'React',
  'Typescript'],
 'missing_preferred_skills': ['Agile',
  'Azure',
  'Excel',
  'Express',
  'Gcp',
  'Github',
  'Linux',
  'Machine Learning',
  'MySQL',
  'Node.js',
  'Product Management'],
 'recommendations': ['Learn or improve Aws',
  'Learn or improve Docker',
  'Learn or improve FastAPI',
  'Learn or improve Git',
  'Learn or improve Kubernetes']}

In [ ]:
# ==============================
# STEP 14: Predict for selected candidate and role
# ==============================

candidate_id = "CAND_920D187A"
selected_role = "AI Engineer"

selected_cv = cv_df[cv_df["candidate_id"] == candidate_id].iloc[0].to_dict()

prediction_result = predict_cv_readiness(selected_cv, selected_role)

prediction_result

{'candidate_id': 'CAND_920D187A',
 'selected_role': 'AI Engineer',
 'predicted_score': 30.45,
 'predicted_level': 'Beginner',
 'matched_required_skills': ['Java', 'NLP', 'Python'],
 'missing_required_skills': ['Aws',
  'Docker',
  'FastAPI',
  'Git',
  'Kubernetes',
  'React',
  'Typescript'],
 'missing_preferred_skills': ['Agile',
  'Azure',
  'Excel',
  'Express',
  'Gcp',
  'Github',
  'Linux',
  'Machine Learning',
  'MySQL',
  'Node.js',
  'Product Management'],
 'recommendations': ['Learn or improve Aws',
  'Learn or improve Docker',
  'Learn or improve FastAPI',
  'Learn or improve Git',
  'Learn or improve Kubernetes']}